**MGMT298D: Science and Strategy of AI**
# Week 1: Linear Regression, Feature Engineering, and Regularization

#### This notebook walks through a complete supervised-learning pipeline for predicting product sales. We load data, split it into train/test sets, fit a baseline regression, engineer richer features, then use regularized models to control overfitting and select the best model via cross-validation.

# 1 Setup & Imports

#### The key packages used in this notebook are `pandas`, a package widely used for managing and organizing tabular data; `numpy`, the foundational library for numerical computing in Python; `matplotlib`, a popular plotting library; and `sklearn` (scikit-learn), the most widely used Python library for machine learning, from which we import regression models, feature scalers, and evaluation metrics.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import (LinearRegression, Lasso, Ridge, ElasticNet,
                                  LassoCV, RidgeCV, ElasticNetCV)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# 1.1 Load & Explore the Data

#### We read the H&M sales data into a `pandas` DataFrame and preview it, then filter down to a single product type so the model focuses on one category at a time.

In [2]:
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df_all = pd.read_csv(url)
print(f"{len(df_all)} rows, {df_all.shape[1]} columns")

# Show all columns in the preview
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df_all.head()

32292 rows, 38 columns


,id,sales,price,name,All over pattern,Denim,Lace,Melange,Solid,Stripe,Beige,Black,Blue,Dark Blue,Dark Grey,Dark Red,Greenish Khaki,Grey,Light Beige,Light Blue,Light Grey,Light Pink,Off White,Pink,Red,White,January,February,March,April,May,June,July,August,September,October,November,December
0,108775015,1388,0.008369,Vest top,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,108775044,513,0.008386,Vest top,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
2,120129001,269,0.016778,Leggings/Tights,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,120129014,70,0.016842,Leggings/Tights,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,146706001,12,0.013316,Bodysuit,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0


#### Filter by Product Type

In [3]:
# Product categories:
# Bag, Belt, Blazer, Blouse, Bodysuit, Boots, Cardigan, Coat, Dress, Hat,
# Hoodie, Jacket, Jumpsuit/Playsuit, Leggings/Tights, Necklace, Other shoe,
# Pumps, Sandals, Scarf, Shirt, Shorts, Skirt, Sneakers, Socks,
# Sunglasses, Sweater, T-shirt, Top, Trousers, Vest top

PRODUCT_TYPE = 'Hoodie'  # Change to any product name

df = df_all[df_all['name'] == PRODUCT_TYPE].copy()
print(f"{PRODUCT_TYPE}: {len(df)} rows, {df['id'].nunique()} products")

Hoodie: 600 rows, 50 products


# 1.2 Train/Test Split

#### Next we split into training and testing datasets. We split by product ID (not by row) so all monthly observations for a given product stay together — 80% of products go to training, 20% are held out for testing.

In [4]:
np.random.seed(42)
product_ids = df['id'].unique()
np.random.shuffle(product_ids)
split = int(0.8 * len(product_ids))
train_ids, test_ids = product_ids[:split], product_ids[split:]
 
print(f"Train: {len(train_ids)} products, Test: {len(test_ids)} products")

Train: 40 products, Test: 10 products


---
# 2 Baseline Linear Regression

#### We choose which columns to use as input features — price, plus the color and pattern indicator columns already in the data.

In [6]:
# Color and pattern indicators are already coded as yes/no columns in the data
color_cols = ['Black', 'Dark Blue', 'White', 'Blue', 'Dark Grey', 'Grey',
              'Light Beige', 'Light Blue', 'Light Pink', 'Beige', 'Dark Red',
              'Greenish Khaki', 'Light Grey', 'Off White', 'Red', 'Pink']
pattern_cols = ['Solid', 'Denim', 'All over pattern', 'Melange', 'Stripe', 'Lace']

basic_features = ['price'] + color_cols + pattern_cols
print(f"Features ({len(basic_features)}): {basic_features}")


Features (23): ['price', 'Black', 'Dark Blue', 'White', 'Blue', 'Dark Grey', 'Grey', 'Light Beige', 'Light Blue', 'Light Pink', 'Beige', 'Dark Red', 'Greenish Khaki', 'Light Grey', 'Off White', 'Red', 'Pink', 'Solid', 'Denim', 'All over pattern', 'Melange', 'Stripe', 'Lace']


In [7]:
def split_and_scale(data, features, train_ids, test_ids):
    """Split by product ID, extract features, and standardize."""
    tr = data[data['id'].isin(train_ids)]
    te = data[data['id'].isin(test_ids)]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(tr[features])
    X_te = scaler.transform(te[features])
    return X_tr, X_te, tr['sales'].values, te['sales'].values, scaler

X_train, X_test, y_train, y_test, scaler1 = split_and_scale(df, basic_features, train_ids, test_ids)

#### Now we fit a plain linear regression on these features and measure how well it predicts on the held-out test set.

In [8]:
# Baseline: plain OLS with price + color + pattern indicators
ols_basic = LinearRegression().fit(X_train, y_train)
mae_basic = mean_absolute_error(y_test, ols_basic.predict(X_test))
print(f"OLS (price + color + pattern) — Test MAE: {mae_basic:.1f}")

OLS (price + color + pattern) — Test MAE: 45.9


---
# 3 Feature Engineering

#### We use `pandas` and `numpy` to create new features from the raw data — lag sales, rolling averages, price transformations, interaction terms, and month indicators. More features give the model more signal, but also increase the risk of overfitting, which motivates regularization in Section 4.

In [ ]:
df_eng = df.copy()

# Lag features: sales from 1, 2, 3 months ago
df_eng['lag_1'] = df_eng.groupby('id')['sales'].shift(1)
df_eng['lag_2'] = df_eng.groupby('id')['sales'].shift(2)
df_eng['lag_3'] = df_eng.groupby('id')['sales'].shift(3)

# Rolling statistics (3-month window, shifted to avoid leakage)
df_eng['ma_3']  = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).mean().shift(1))
df_eng['std_3'] = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).std().shift(1))

# Price features
df_eng['price_pct_change'] = df_eng.groupby('id')['price'].pct_change()
df_eng['price_sq'] = df_eng['price'] ** 2

# Interaction terms
df_eng['price_x_lag_1'] = df_eng['price'] * df_eng['lag_1']
df_eng['lag1_x_lag2']   = df_eng['lag_1'] * df_eng['lag_2']

df_eng.fillna(0, inplace=True)
df_eng.replace([np.inf, -np.inf], 0, inplace=True)

# Month indicators (already coded as yes/no columns in the data)
month_cols = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']

# Engineered features + baseline features
engineered = ['price_sq', 'price_pct_change',
              'lag_1', 'lag_2', 'lag_3', 'ma_3', 'std_3',
              'price_x_lag_1', 'lag1_x_lag2'] + month_cols
all_features = basic_features + engineered
print(f"{len(all_features)} features ({len(basic_features)} baseline + {len(engineered)} engineered)")

In [10]:
X_train2, X_test2, y_train2, y_test2, scaler2 = split_and_scale(df_eng, all_features, train_ids, test_ids)

# OLS with all engineered features — likely overfits
ols_eng = LinearRegression().fit(X_train2, y_train2)
mae_eng_train = mean_absolute_error(y_train2, ols_eng.predict(X_train2))
mae_eng_test  = mean_absolute_error(y_test2, ols_eng.predict(X_test2))

print(f"OLS ({len(all_features)} features)")
print(f"  Train MAE: {mae_eng_train:.1f}")
print(f"  Test  MAE: {mae_eng_test:.1f}")
print(f"  Gap: {mae_eng_test - mae_eng_train:.1f}  (large = overfitting)")

OLS (44 features)
  Train MAE: 32.1
  Test  MAE: 37.1
  Gap: 5.0  (large = overfitting)


---
# 4 Regularization

#### Plain OLS overfits with many features. Here we apply `Lasso`, `Ridge`, and `ElasticNet` from `sklearn` — each adds a penalty controlled by λ that shrinks weights and improves generalization. We sweep a grid of λ values to see how penalty strength trades off between underfitting and overfitting.

#### Lasso shrinks and eliminates weights — as λ increases, more weights are pushed to exactly zero. Ridge shrinks weights but keeps them all — it never zeros out any coefficient.

In [11]:
# Sweep λ values for Lasso and Ridge
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
results = []

for lam in lambdas:
    for name, model in [('Lasso', Lasso(alpha=lam, max_iter=10000)),
                        ('Ridge', Ridge(alpha=lam))]:
        m = model.fit(X_train2, y_train2)
        results.append({
            'Model': name, 'λ': lam,
            'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
            'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
            'nonzero':   int(np.sum(m.coef_ != 0))
        })

results_df = pd.DataFrame(results)
print("LASSO")
print(results_df[results_df['Model']=='Lasso'].drop(columns='Model').to_string(index=False))
print(f"\nRIDGE")
print(results_df[results_df['Model']=='Ridge'].drop(columns='Model').to_string(index=False))

LASSO
      λ  Train MAE  Test MAE  nonzero
   0.01  32.037185 37.022551       31
   0.10  31.747766 36.185796       30
   1.00  30.721263 33.244116       24
  10.00  34.588299 38.774360        6
 100.00  46.824340 44.720347        0
1000.00  46.824340 44.720347        0

RIDGE
      λ  Train MAE  Test MAE  nonzero
   0.01  32.070974 37.114920       33
   0.10  32.052809 37.055404       33
   1.00  31.894051 36.540509       33
  10.00  31.400477 34.677709       33
 100.00  31.116613 31.841725       33
1000.00  35.305688 33.648555       33


# 5 Model Selection via Cross-Validation

#### Instead of manually picking λ, we use `LassoCV`, `RidgeCV`, and `ElasticNetCV` from `sklearn` — these try many λ values using cross-validation on the training set and automatically select the one that minimizes error.

In [12]:
# Shared penalty grid for all three CV models
lambdas_cv = np.logspace(-3, 3, 50)

# Use cross-validation to select best λ for Lasso and Ridge, best λ + α for Elastic Net
lasso_cv = LassoCV(alphas=lambdas_cv, max_iter=10000).fit(X_train2, y_train2)
ridge_cv = RidgeCV(alphas=lambdas_cv).fit(X_train2, y_train2)
enet_cv  = ElasticNetCV(alphas=lambdas_cv, max_iter=10000).fit(X_train2, y_train2)

# Compare all models
print(f"OLS (price + color + pattern)  — Test MAE: {mae_basic:.1f}")
print(f"OLS ({len(all_features)} features)    — Test MAE: {mae_eng_test:.1f}")
print(f"Lasso (λ={lasso_cv.alpha_:.3f})   — Test MAE: {mean_absolute_error(y_test2, lasso_cv.predict(X_test2)):.1f}  — {int(np.sum(lasso_cv.coef_ != 0))}/{len(all_features)} features")
print(f"Ridge (λ={ridge_cv.alpha_:.3f})   — Test MAE: {mean_absolute_error(y_test2, ridge_cv.predict(X_test2)):.1f}  — {int(np.sum(ridge_cv.coef_ != 0))}/{len(all_features)} features")
print(f"ElasticNet (λ={enet_cv.alpha_:.3f}, α={enet_cv.l1_ratio_:.2f}) — Test MAE: {mean_absolute_error(y_test2, enet_cv.predict(X_test2)):.1f}  — {int(np.sum(enet_cv.coef_ != 0))}/{len(all_features)} features")

OLS (price + color + pattern)  — Test MAE: 45.9
OLS (44 features)    — Test MAE: 37.1
Lasso (λ=6.251)   — Test MAE: 37.1  — 10/44 features
Ridge (λ=429.193)   — Test MAE: 31.4  — 33/44 features
ElasticNet (λ=1.526, α=0.50) — Test MAE: 31.7  — 28/44 features
